# Startup Investment Outcome — Preprocessing Pipeline

Each step below implements a decision documented in `01_EDA.ipynb`.
The entire preprocessing is wrapped in a **sklearn Pipeline** that is fit on the training set only,
then applied identically to the validation and test sets.

In [121]:
import pandas as pd
import numpy as np
import os
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [122]:
df = pd.read_csv('data/investments_VC_clean.csv')
print(f"Loaded: {df.shape}")
df.head(3)

Loaded: (39802, 39)


,permalink,name,homepage_url,category_list,market,funding_total_usd,status,country_code,state_code,region,...,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H
0,/organization/waywire,#waywire,http://www.waywire.com,|Entertainment|Politics|Social Media|News|,News,1750000.0,acquired,USA,NY,New York City,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,/organization/tv-communications,&TV Communications,http://enjoyandtv.com,|Games|,Games,4000000.0,operating,USA,CA,Los Angeles,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,/organization/rock-your-paper,'Rock' Your Paper,http://www.rockyourpaper.org,|Publishing|Education|,Publishing,40000.0,operating,EST,NaN,Tallinn,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 1. Stratified Train / Val / Test Split (70 / 15 / 15)

In [123]:
# Encode target
status_map = {'closed': 0, 'operating': 1, 'acquired': 2}
df['status_code'] = df['status'].map(status_map)

X = df.drop(columns=['status_code'])
y = df['status_code']

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15/0.85, stratify=y_train_val, random_state=42)

print(f"Train:      {X_train.shape[0]:,} rows")
print(f"Validation: {X_val.shape[0]:,} rows")
print(f"Test:       {X_test.shape[0]:,} rows")

for name, y_s in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    pct = y_s.value_counts(normalize=True).sort_index() * 100
    print(f"{name}: closed={pct.get(0,0):.1f}%  operating={pct.get(1,0):.1f}%  acquired={pct.get(2,0):.1f}%")

os.makedirs('data', exist_ok=True)
X_train.assign(status_code=y_train).to_csv('data/train_set.csv', index=False)
X_val.assign(status_code=y_val).to_csv('data/validation_set.csv', index=False)
X_test.assign(status_code=y_test).to_csv('data/test_set.csv', index=False)
print("\nRaw splits saved to data/")

Train:      27,860 rows
Validation: 5,971 rows
Test:       5,971 rows
Train: closed=5.4%  operating=86.5%  acquired=8.1%
Val: closed=5.4%  operating=86.5%  acquired=8.1%
Test: closed=5.4%  operating=86.5%  acquired=8.1%

Raw splits saved to data/


## 2. Custom Transformers

Each class implements one EDA decision. All fit on train only.

> **Note (pandas 2.x):** `pd.read_csv` infers text columns as `StringDtype`, which rejects numeric assignments.
> `DTypeConverter` runs first and converts all known numeric columns to `float64` by dropping and re-adding them.

In [124]:
# Step 0 — pandas 2.x StringDtype fix
# Drops and recreates numeric columns as float64 so all later transformers can assign freely.
class DTypeConverter(BaseEstimator, TransformerMixin):
    NUMERIC = [
        'funding_total_usd', 'funding_rounds',
        'seed', 'venture', 'angel', 'grant', 'private_equity', 'debt_financing',
        'equity_crowdfunding', 'convertible_note', 'undisclosed', 'product_crowdfunding',
        'post_ipo_equity', 'post_ipo_debt', 'secondary_market',
        'round_A', 'round_B', 'round_C', 'round_D', 'round_E',
        'round_F', 'round_G', 'round_H',
        'founded_year', 'founded_month', 'founded_quarter',
    ]
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        converted = {
            col: pd.to_numeric(X[col].astype(object), errors='coerce')
            for col in self.NUMERIC if col in X.columns
        }
        # Drop StringDtype columns, re-add as float64
        X = X.drop(columns=list(converted.keys()))
        for col, vals in converted.items():
            X[col] = vals
        return X

In [125]:
# EDA §3.2 — Fill founded_year/month/quarter nulls from founded_at
# (DTypeConverter already ensured these are float64, so simple fillna works)
class DateFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        at = pd.to_datetime(X['founded_at'], errors='coerce')
        X['founded_year']    = X['founded_year'].fillna(at.dt.year)
        X['founded_month']   = X['founded_month'].fillna(at.dt.month)
        X['founded_quarter'] = X['founded_quarter'].fillna(at.dt.quarter)
        return X

In [126]:
# EDA §3.3 — Fill null market with "Unknown"
class MarketFiller(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        X['market'] = X['market'].fillna('Unknown')
        return X

In [127]:
# EDA §3.4 — Fill country_code nulls with "Unknown"
# (analysis showed 0 rows where state_code could recover country_code)
class CountryFiller(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        X['country_code'] = X['country_code'].fillna('Unknown')
        return X

In [128]:
# EDA §3.4 — Create geo_cluster feature
class GeoClusterCreator(BaseEstimator, TransformerMixin):
    SV  = {'San Francisco','San Jose','Palo Alto','Mountain View','Menlo Park',
            'Redwood City','Sunnyvale','Santa Clara','Cupertino','San Mateo','Burlingame'}
    NY  = {'New York','Brooklyn','Manhattan','New York City'}
    BOS = {'Boston','Cambridge','Somerville','Waltham'}
    SEA = {'Seattle','Bellevue','Redmond','Kirkland'}
    LA  = {'Los Angeles','Santa Monica','Venice','Culver City','Pasadena'}

    def fit(self, X, y=None): return self
    def _cluster(self, row):
        cc, city = row.get('country_code',''), str(row.get('city',''))
        if pd.isna(cc) or cc == 'Unknown': return 'Unknown'
        if cc != 'USA': return cc
        if city in self.SV:  return 'USA-SiliconValley'
        if city in self.NY:  return 'USA-NY'
        if city in self.BOS: return 'USA-Boston'
        if city in self.SEA: return 'USA-Seattle'
        if city in self.LA:  return 'USA-LA'
        return 'USA-Other'
    def transform(self, X):
        X = X.copy()
        X['geo_cluster'] = X.apply(self._cluster, axis=1)
        return X

In [129]:
# EDA §4 — log1p transform on all funding amount columns
class FundingLogTransformer(BaseEstimator, TransformerMixin):
    COLS = ['funding_total_usd','seed','venture','angel','grant','private_equity',
            'debt_financing','equity_crowdfunding','convertible_note','undisclosed',
            'product_crowdfunding','post_ipo_equity','post_ipo_debt','secondary_market']
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        for col in self.COLS:
            if col in X.columns:
                X[f'log_{col}'] = np.log1p(X[col].fillna(0))
        return X

In [130]:
# EDA §5 — Binary flags for rounds A/B/C; sum of D–H into round_D_plus; drop round_D through round_H
class RoundBinarizer(BaseEstimator, TransformerMixin):
    EARLY = ['round_A', 'round_B', 'round_C']
    LATE  = ['round_D', 'round_E', 'round_F', 'round_G', 'round_H']

    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        for col in self.EARLY:
            if col in X.columns:
                X[f'has_{col}'] = (X[col].fillna(0) > 0).astype(int)
        late_cols = [c for c in self.LATE if c in X.columns]
        X['round_D_plus'] = X[late_cols].fillna(0).sum(axis=1)
        X = X.drop(columns=late_cols)
        return X

In [131]:
# EDA §6 — days_to_first_funding + median imputation
class FundingTimelineCalculator(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.median_ = None

    def fit(self, X, y=None):
        fa  = pd.to_datetime(X['founded_at'], errors='coerce')
        ffa = pd.to_datetime(X['first_funding_at'], errors='coerce')
        self.median_ = (ffa - fa).dt.days.median()
        return self

    def transform(self, X):
        X = X.copy()
        fa  = pd.to_datetime(X['founded_at'], errors='coerce')
        ffa = pd.to_datetime(X['first_funding_at'], errors='coerce')
        X['days_to_first_funding'] = (ffa - fa).dt.days.fillna(self.median_)
        return X

In [132]:
# EDA §7 — Target encoding for market and country_code
class TargetEncoder(BaseEstimator, TransformerMixin):
    CATS = ['market', 'country_code']

    def fit(self, X, y=None):
        self.encodings_ = {}
        self.global_means_ = {}
        if y is None: return self
        df_fit = X.copy()
        df_fit['_target'] = y.values if hasattr(y, 'values') else np.array(y)
        df_fit['_acquired'] = (df_fit['_target'] == 2).astype(int)
        df_fit['_closed']   = (df_fit['_target'] == 0).astype(int)
        for cat in self.CATS:
            if cat not in df_fit.columns: continue
            self.global_means_[cat] = {
                'acquired': df_fit['_acquired'].mean(),
                'closed':   df_fit['_closed'].mean()
            }
            self.encodings_[cat] = {
                'acquired': df_fit.groupby(cat)['_acquired'].mean(),
                'closed':   df_fit.groupby(cat)['_closed'].mean()
            }
        return self

    def transform(self, X):
        X = X.copy()
        for cat in self.CATS:
            if cat not in X.columns or cat not in self.encodings_: continue
            for outcome in ['acquired', 'closed']:
                mapping = self.encodings_[cat][outcome]
                fallback = self.global_means_[cat][outcome]
                X[f'{cat}_enc_{outcome}'] = X[cat].map(mapping).fillna(fallback)
            X = X.drop(columns=[cat])
        return X

In [133]:
# EDA §3.2 — Median imputation for remaining founded_year/month/quarter nulls
# KNNImputer was too memory-intensive (~1 GB) at this data scale → replaced with per-column median
class DateMedianImputer(BaseEstimator, TransformerMixin):
    DATE_COLS = ['founded_year', 'founded_month', 'founded_quarter']

    def fit(self, X, y=None):
        self.medians_ = {col: X[col].median() for col in self.DATE_COLS if col in X.columns}
        return self

    def transform(self, X):
        X = X.copy()
        for col, med in self.medians_.items():
            if col in X.columns:
                X[col] = X[col].fillna(med)
        return X

In [134]:
# Drop columns not needed for modeling
class ColumnDropper(BaseEstimator, TransformerMixin):
    DROP = ['permalink','name','homepage_url','category_list',
            'status','state_code','region','city','status_label',
            'founded_at','first_funding_at','last_funding_at']
    def fit(self, X, y=None):
        self.fitted_ = True
        return self
    def transform(self, X):
        X = X.copy()
        to_drop = [c for c in self.DROP if c in X.columns]
        return X.drop(columns=to_drop)

## 3. Assemble and Run Pipeline

In [135]:
pipeline = Pipeline([
    ('dtype_converter',     DTypeConverter()),
    ('date_extractor',      DateFeatureExtractor()),
    ('country_filler',      CountryFiller()),
    ('market_filler',       MarketFiller()),
    ('geo_cluster',         GeoClusterCreator()),
    ('log_transformer',     FundingLogTransformer()),
    ('round_binarizer',     RoundBinarizer()),
    ('timeline_calculator', FundingTimelineCalculator()),
    ('target_encoder',      TargetEncoder()),
    ('date_median_imputer', DateMedianImputer()),
    ('column_dropper',      ColumnDropper()),
])

print("Pipeline steps:")
for name, step in pipeline.steps:
    print(f"  {name:25s} → {step.__class__.__name__}")

Pipeline steps:
  dtype_converter           → DTypeConverter
  date_extractor            → DateFeatureExtractor
  country_filler            → CountryFiller
  market_filler             → MarketFiller
  geo_cluster               → GeoClusterCreator
  log_transformer           → FundingLogTransformer
  round_binarizer           → RoundBinarizer
  timeline_calculator       → FundingTimelineCalculator
  target_encoder            → TargetEncoder
  date_median_imputer       → DateMedianImputer
  column_dropper            → ColumnDropper


In [136]:
pipeline.fit(X_train, y_train)
print("Pipeline fitted on training set.")

X_train_proc = pipeline.transform(X_train)
X_val_proc   = pipeline.transform(X_val)
X_test_proc  = pipeline.transform(X_test)
print(f"Train processed: {X_train_proc.shape}")
print(f"Val   processed: {X_val_proc.shape}")
print(f"Test  processed: {X_test_proc.shape}")

Pipeline fitted on training set.
Train processed: (27860, 45)
Val   processed: (5971, 45)
Test  processed: (5971, 45)


## 4. Verification

In [137]:
print("=== Shapes ===")
for name, X_p in [('Train', X_train_proc), ('Val', X_val_proc), ('Test', X_test_proc)]:
    print(f"{name}: {X_p.shape}")

=== Shapes ===
Train: (27860, 45)
Val: (5971, 45)
Test: (5971, 45)


In [138]:
print("=== Null counts after pipeline ===")
for name, X_p in [('Train', X_train_proc), ('Val', X_val_proc), ('Test', X_test_proc)]:
    nulls = X_p.isnull().sum().sum()
    print(f"{name}: {nulls} total nulls")

=== Null counts after pipeline ===
Train: 0 total nulls
Val: 0 total nulls
Test: 0 total nulls


In [139]:
# Which columns have nulls, and how many?
null_summary = X_train_proc.isnull().sum()
null_summary = null_summary[null_summary > 0].sort_values(ascending=False)
pct = (null_summary / len(X_train_proc) * 100).round(2)
print("Columns with nulls after pipeline (train):")
print(pd.DataFrame({'null_count': null_summary, 'null_pct': pct}).to_string())

Columns with nulls after pipeline (train):
Empty DataFrame
Columns: [null_count, null_pct]
Index: []


In [140]:
# Are the nulls concentrated in the same rows, or spread across different rows?
null_mask = X_train_proc.isnull().any(axis=1)
print(f"Rows with at least one null: {null_mask.sum():,} ({null_mask.mean()*100:.1f}%)")

# Show co-occurrence: which columns tend to be null together?
null_cols = X_train_proc.columns[X_train_proc.isnull().any()].tolist()
if null_cols:
    print(f"\nNull columns: {null_cols}")
    print("\nCo-occurrence matrix (how often two columns are null in the same row):")
    cooc = X_train_proc[null_cols].isnull().astype(int).T.dot(
           X_train_proc[null_cols].isnull().astype(int))
    print(cooc.to_string())
else:
    print("No nulls found — pipeline is clean!")

Rows with at least one null: 0 (0.0%)
No nulls found — pipeline is clean!


In [141]:
# For each null column: trace back which pipeline step should have handled it
# and show sample raw values to understand why it slipped through
if null_cols:
    for col in null_cols:
        null_rows = X_train_proc[X_train_proc[col].isnull()].index
        print(f"\n--- {col}: {len(null_rows)} nulls ---")
        # Show corresponding raw values from X_train
        if col in X_train.columns:
            print(f"  Raw values in X_train for null rows:")
            print(f"  {X_train.loc[null_rows, col].value_counts(dropna=False).head(5).to_dict()}")
        else:
            print(f"  (derived column — not in raw X_train)")
else:
    print("No nulls to trace.")

No nulls to trace.


In [142]:
print("=== Class distribution preserved? ===")
for name, y_s in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    pct = y_s.value_counts(normalize=True).sort_index() * 100
    print(f"{name}: closed={pct.get(0,0):.1f}%  operating={pct.get(1,0):.1f}%  acquired={pct.get(2,0):.1f}%")

=== Class distribution preserved? ===
Train: closed=5.4%  operating=86.5%  acquired=8.1%
Val: closed=5.4%  operating=86.5%  acquired=8.1%
Test: closed=5.4%  operating=86.5%  acquired=8.1%


In [143]:
print("=== Final feature list ===")
print(list(X_train_proc.columns))
print(f"\nTotal features: {X_train_proc.shape[1]}")

=== Final feature list ===
['funding_total_usd', 'funding_rounds', 'seed', 'venture', 'angel', 'grant', 'private_equity', 'debt_financing', 'equity_crowdfunding', 'convertible_note', 'undisclosed', 'product_crowdfunding', 'post_ipo_equity', 'post_ipo_debt', 'secondary_market', 'round_A', 'round_B', 'round_C', 'founded_year', 'founded_month', 'founded_quarter', 'geo_cluster', 'log_funding_total_usd', 'log_seed', 'log_venture', 'log_angel', 'log_grant', 'log_private_equity', 'log_debt_financing', 'log_equity_crowdfunding', 'log_convertible_note', 'log_undisclosed', 'log_product_crowdfunding', 'log_post_ipo_equity', 'log_post_ipo_debt', 'log_secondary_market', 'has_round_A', 'has_round_B', 'has_round_C', 'round_D_plus', 'days_to_first_funding', 'market_enc_acquired', 'market_enc_closed', 'country_code_enc_acquired', 'country_code_enc_closed']

Total features: 45


In [144]:
X_train_proc.assign(status_code=y_train.values).to_csv('data/train_processed.csv', index=False)
X_val_proc.assign(status_code=y_val.values).to_csv('data/val_processed.csv', index=False)
X_test_proc.assign(status_code=y_test.values).to_csv('data/test_processed.csv', index=False)
print("Saved: data/train_processed.csv, val_processed.csv, test_processed.csv")

Saved: data/train_processed.csv, val_processed.csv, test_processed.csv
